# Gibbs Sampler for Gaussian Spatial Models

This notebook demonstrates the **block-Gibbs sampler** for Gaussian spatial regression models (SAR, SEM, SDM, SDEM) in `bayespecon`. The Gibbs sampler exploits conditional conjugacy to avoid the poor NUTS geometry that spatial Jacobians create, often yielding **10–100× faster** sampling for large datasets.

## When to Use Gibbs vs NUTS

| Criterion | NUTS (default) | Gibbs |
|---|---|---|
| **Model type** | Any (Gaussian, Student-t, NB, Probit, Tobit) | Gaussian only (SAR, SEM, SDM, SDEM) |
| **Posterior geometry** | Handles banana/funnel shapes via adaptation | Avoids banana geometry via conjugacy |
| **Speed** | Slow for large $n$ (gradient through Jacobian) | Fast (direct draws for β, σ²; 1-D slice/MALA for ρ/λ) |
| **Robust models** | ✅ Student-t errors | ❌ Not supported |
| **Non-Gaussian** | ✅ NB, Probit, Tobit | ❌ Not supported |
| **ESS/s** | Low for spatial models | High (conjugate blocks mix well) |
| **Tuning** | 1000+ tuning steps needed | Minimal (adaptive slice width or MALA step size) |

**Rule of thumb**: Use `sampler='gibbs'` for Gaussian spatial models with $n > 100$. Use NUTS (default) for non-Gaussian models or when you need Student-t robustness.

In [18]:
%load_ext autoreload
%autoreload 2

import arviz as az
import geopandas as gpd
import libpysal
import numpy as np

from bayespecon.models import SAR, SDEM, SDM, SEM

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data: Columbus Crime Dataset

We use the classic Columbus (OH) neighbourhood crime dataset from `libpysal`.

In [19]:
# Load Columbus dataset
gdf = gpd.read_file(libpysal.examples.get_path("columbus.shp"))

# Response: crime rate
y = gdf["CRIME"].values.astype(float)

# Covariates: income + housing value
X = np.column_stack(
    [
        np.ones(len(y)),
        gdf["INC"].values.astype(float),
        gdf["HOVAL"].values.astype(float),
    ]
)

# Spatial weights: queen contiguity
W = libpysal.graph.Graph.build_contiguity(gdf)
W = W.transform("r")  # row-standardise

print(f"n = {len(y)}, k = {X.shape[1]}")

n = 49, k = 3


## SAR Model: Gibbs vs NUTS

The SAR (Spatial Autoregressive) model is:

$$y = \rho W y + X\beta + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2 I)$$

The Gibbs sampler uses a **3-block** strategy:
1. **β | ρ, σ², y** — conjugate normal (direct draw)
2. **σ² | β, ρ, y** — conjugate inverse-gamma (direct draw)
3. **ρ | β, σ², y** — 1-D slice sampling or MALA (non-conjugate, scalar)

Only ρ is non-conjugate, and it's a scalar — making the update trivial.

In [20]:
# --- NUTS (default) ---
model_nuts = SAR(y=y, X=X, W=W)
idata_nuts = model_nuts.fit(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
    idata_kwargs={"log_likelihood": True},
)

# --- Gibbs (numpy path) ---
model_gibbs = SAR(y=y, X=X, W=W)
idata_gibbs = model_gibbs.fit(
    sampler="gibbs",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=1,
    idata_kwargs={"log_likelihood": True},
)

print("NUTS posterior means:")
print(f"  rho   = {float(idata_nuts.posterior['rho'].mean()):.4f}")
print(f"  beta  = {idata_nuts.posterior['beta'].mean(dim=['chain', 'draw']).values}")
print(f"  sigma = {float(idata_nuts.posterior['sigma'].mean()):.4f}")
print()
print("Gibbs posterior means:")
print(f"  rho   = {float(idata_gibbs.posterior['rho'].mean()):.4f}")
print(f"  beta  = {idata_gibbs.posterior['beta'].mean(dim=['chain', 'draw']).values}")
print(f"  sigma = {float(idata_gibbs.posterior['sigma'].mean()):.4f}")

TypeError: GaussianGibbsPriors.__init__() got an unexpected keyword argument 'sigma2_alpha'

In [ ]:
az.summary(idata_nuts)["ess_bulk"]

In [ ]:
az.summary(idata_gibbs)["ess_bulk"]

In [ ]:
model_gibbs.spatial_diagnostics_decision()

## Gibbs Backend Options

The Gibbs sampler supports two execution backends:

| Backend | `gibbs_method` | ρ/λ update | JIT compiled | Requires |
|---|---|---|---|---|
| **NumPy** | `"numpy"` (default) | Adaptive slice sampling | No | NumPy only |
| **JAX** | `"jax"` | MALA (gradient-guided) or RW-MH | Yes (`@eqx.filter_jit`) | JAX + equinox |

### NumPy path (default)

The NumPy path uses adaptive slice sampling for ρ/λ. It's pure Python/NumPy — no JAX dependency. Best for moderate $n$ (up to ~1000). Chains run sequentially by default (`chain_method="sequential"`).

### JAX path

The JAX path compiles the entire 3-block Gibbs step into a single XLA kernel via `@eqx.filter_jit`. It uses MALA (Metropolis-adjusted Langevin algorithm) for ρ/λ by default, which leverages `jax.value_and_grad` on the collapsed log-density for gradient-guided proposals. Best for large $n$ where JIT compilation amortizes over many iterations. Chains run in parallel via `jax.vmap` by default (`chain_method="vectorized"`).

In [ ]:
# JAX Gibbs path with MALA
model_jax = SAR(y=y, X=X, W=W)
idata_jax = model_jax.fit(
    sampler="gibbs",
    gibbs_method="jax",  # JAX JIT backend (defaults to chain_method="vectorized")
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    use_mala=True,
    use_slice=False,
)

# JAX Gibbs path with slice
model_rw = SAR(y=y, X=X, W=W)
idata_rw = model_rw.fit(
    sampler="gibbs",
    gibbs_method="jax",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
)

print("JAX MALA posterior means:")
print(f"  rho   = {float(idata_jax.posterior['rho'].mean()):.4f}")
print(f"  sigma = {float(idata_jax.posterior['sigma'].mean()):.4f}")
print()
print("JAX Slice posterior means:")
print(f"  rho   = {float(idata_rw.posterior['rho'].mean()):.4f}")
print(f"  sigma = {float(idata_rw.posterior['sigma'].mean()):.4f}")

## Log-Determinant Method Choices

The spatial Jacobian $\log|I - \rho W|$ is the computational bottleneck. `bayespecon` supports several approximation methods:

| Method | `logdet_method` | Extra option | Precompute | Per-ρ cost | JIT-compatible | Best for |
|---|---|---|---|---|---|---|
| Eigenvalue | `"eigenvalue"` | — | O(n³) | O(n) | ✅ | n < 500 |
| Chebyshev | `"chebyshev"` | exact coefficients when feasible | O(n³) or O(nm) | O(m) | ✅ | default for n ≥ 500 |
| Chebyshev + stochastic traces | `"chebyshev"` | `trace_estimator="hutchpp"` (default), `"hutchinson"`, or `"xtrace"` | O(kmn) | O(m) | ✅ | large sparse W |

The Gibbs sampler automatically uses the same logdet configuration as the model. For the JAX path, the supported public choices are JIT-compatible.

In [ ]:
# Chebyshev logdet with JAX Gibbs
model_cheb = SAR(y=y, X=X, W=W, logdet_method="chebyshev")
idata_cheb = model_cheb.fit(
    sampler="gibbs",
    gibbs_method="jax",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
)

print(
    f"Chebyshev logdet + JAX Gibbs: rho = {float(idata_cheb.posterior['rho'].mean()):.4f}"
)

## SEM, SDM, and SDEM Models

The Gibbs sampler works for all four Gaussian spatial model types:

| Model | Spatial parameter | ρ/λ update | Null model for ρ/λ |
|---|---|---|---|
| SAR | ρ (lag) | Collapsed slice/MALA | Integrates out β, σ² |
| SEM | λ (error) | Conditional slice/MALA | Conditional on β, σ² |
| SDM | ρ (lag) | Collapsed slice/MALA | Same as SAR, Z = [X, WX] |
| SDEM | λ (error) | Conditional slice/MALA | Same as SEM, Z = [X, WX] |

In [ ]:
# SEM Gibbs
model_sem = SEM(y=y, X=X, W=W)
idata_sem = model_sem.fit(
    sampler="gibbs",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
)
print(
    f"SEM Gibbs: lam = {float(idata_sem.posterior['lam'].mean()):.4f}, "
    f"sigma = {float(idata_sem.posterior['sigma'].mean()):.4f}"
)

# SDM Gibbs
model_sdm = SDM(y=y, X=X, W=W)
idata_sdm = model_sdm.fit(
    sampler="gibbs",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
)
print(
    f"SDM Gibbs: rho = {float(idata_sdm.posterior['rho'].mean()):.4f}, "
    f"beta shape = {idata_sdm.posterior['beta'].shape}"
)

# SDEM Gibbs
model_sdem = SDEM(y=y, X=X, W=W)
idata_sdem = model_sdem.fit(
    sampler="gibbs",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
    chain_method="parallel",
)
print(
    f"SDEM Gibbs: lam = {float(idata_sdem.posterior['lam'].mean()):.4f}, "
    f"beta shape = {idata_sdem.posterior['beta'].shape}"
)

In [ ]:
az.summary(idata_sdm)

## InferenceData Compatibility

The Gibbs sampler produces the same `az.InferenceData` output as NUTS, with `posterior`, `log_likelihood`, and `observed_data` groups. This means all ArviZ diagnostics (LOO, WAIC, summary, etc.) work seamlessly.

In [ ]:
# ArviZ diagnostics work with Gibbs-produced InferenceData
print("Groups:", idata_gibbs.groups())
print()

# LOO cross-validation
loo = az.loo(idata_gibbs)
print(f"LOO: elpd = {loo.elpd_loo:.2f}, SE = {loo.se:.2f}")
print()

# WAIC
waic = az.waic(idata_gibbs)
print(f"WAIC: elpd = {waic.elpd_waic:.2f}, SE = {waic.se:.2f}")
print()

# Summary
az.summary(idata_gibbs, var_names=["rho", "sigma"])

In [ ]:
az.plot_trace(idata_gibbs)

## Spatial Diagnostics

The Gibbs-produced `InferenceData` also works with `bayespecon`'s Bayesian LM diagnostics, which test for spatial dependence in the residuals.

In [ ]:
# Run spatial diagnostics on the Gibbs-fitted SAR model
report = model_gibbs.spatial_diagnostics()
print(report)

## Summary

The block-Gibbs sampler for Gaussian spatial models in `bayespecon` provides:

1. **Same output format** as NUTS (`az.InferenceData` with `posterior`, `log_likelihood`, `observed_data`)
2. **Two backends**: NumPy (adaptive slice, `n_jobs` controls parallelism) and JAX (MALA or RW-MH, `chain_method` controls parallelism)
3. **All four Gaussian models**: SAR, SEM, SDM, SDEM
4. **Full ArviZ compatibility**: LOO, WAIC, summary, plot
5. **Full diagnostics compatibility**: Bayesian LM tests, spatial diagnostics

```python
# Quick reference
model = SAR(y=y, X=X, W=W)

# NumPy Gibbs (default, n_jobs=-1 runs chains in parallel via joblib)
idata = model.fit(sampler="gibbs", draws=2000, tune=1000, chains=4)

# NumPy Gibbs with sequential chains (for debugging)
idata = model.fit(sampler="gibbs", draws=2000, tune=1000, chains=4, n_jobs=1)

# JAX Gibbs with MALA (chains run in parallel via vmap by default)
idata = model.fit(sampler="gibbs", gibbs_method="jax", draws=2000, tune=1000, chains=4)

# JAX Gibbs with RW-MH
idata = model.fit(sampler="gibbs", gibbs_method="jax", use_mala=False, draws=2000, tune=1000, chains=4)

# Chebyshev logdet + JAX Gibbs
model = SAR(y=y, X=X, W=W, logdet_method="chebyshev")
idata = model.fit(sampler="gibbs", gibbs_method="jax", draws=2000, tune=1000, chains=4)
```